In [ ]:
from typing import TypedDict
from IPython.display import Image, display
from langgraph.graph import StateGraph, START,END

class PaymentState(TypedDict):
    initial_price: float
    coupon_amount: float
    discount_amount: float
    tax_rate: float
    price_after_coupon: float
    price_after_discount: float
    price_final: float

def apply_coupon(state: PaymentState) -> PaymentState:
    state['price_after_coupon'] = state['initial_price'] - state['coupon_amount'] # Apply coupon
    return state

def apply_discount(state: PaymentState) -> PaymentState:
    state['price_after_discount'] = state['price_after_coupon'] - state['discount_amount']
    return state

def apply_tax(state: PaymentState) -> PaymentState:
    state['price_final'] = state['price_after_discount'] * (1 + state['tax_rate'])  # Apply tax
    return state


In [ ]:
# build graph by connecting nodes by edges
builder = StateGraph(PaymentState)

builder.add_node("apply_discount", apply_discount)
builder.add_node("apply_coupon", apply_coupon)
builder.add_node("apply_tax", apply_tax)

builder.add_edge(START, "apply_coupon")
builder.add_edge("apply_coupon", "apply_discount")
builder.add_edge("apply_discount", "apply_tax")
builder.add_edge("apply_tax", END)

graph = builder.compile()

In [ ]:
# display the graph
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# invoke the graph with initial state
graph.invoke({"initial_price": 1000.0, "coupon_amount":200, "discount_amount": 100, "tax_rate": 0.10})